In [61]:
import os
import shutil
import xml.etree.ElementTree as ET
from random import shuffle

### Source and destination paths

In [62]:
source_paths = [
    r"E:\image\CarLicense\images",   # Other dataset
    r"E:\image\CarLicense\car"       # Iranian dataset
]

output_base = r"E:\image\CarLicense\licenses_cars"

### Train/validation split ratios

In [63]:
train_ratio = 0.8
val_ratio = 0.2

In [64]:
# class names (add more if you need)
classes = ["car","licence"]

In [65]:
def voc_to_yolo(xml_file, output_folder):
    """تبدیل annotation از VOC XML به YOLO TXT"""
    tree = ET.parse(xml_file)
    root = tree.getroot()

    img_w = int(root.find("size/width").text)
    img_h = int(root.find("size/height").text)

    yolo_lines = []
    for obj in root.findall("object"):
        cls = obj.find("name").text
        if cls not in classes:
            continue
        cls_id = classes.index(cls)

        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        # YOLO format
        x_center = (xmin + xmax) / 2 / img_w
        y_center = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h

        yolo_lines.append(f"{cls_id} {x_center} {y_center} {w} {h}")

    os.makedirs(output_folder, exist_ok=True)
    txt_name = os.path.splitext(os.path.basename(xml_file))[0] + ".txt"
    with open(os.path.join(output_folder, txt_name), "w") as f:
        f.write("\n".join(yolo_lines))

### Create directories if they don't exist

In [66]:
# for path in [train_image_path, train_label_path, val_image_path, val_label_path]:
#     os.makedirs(path, exist_ok=True)

### Collect all image files

In [67]:
def gather_files(source_paths):
    """Find all images and labels"""
    images, labels_xml, labels_txt = [], [], []
    for path in source_paths:
        for dirname, _, files in os.walk(path):
            for filename in files:
                if filename.endswith(('.jpg', '.png')):
                    images.append(os.path.join(dirname, filename))
                elif filename.endswith('.xml'):
                    labels_xml.append(os.path.join(dirname, filename))
                elif filename.endswith('.txt'):
                    labels_txt.append(os.path.join(dirname, filename))
    return images, labels_xml, labels_txt


In [68]:
def prepare_labels(xmls, labels_dir):
    """Convert all XML labels to YOLO TXT"""
    for xml in xmls:
        voc_to_yolo(xml, labels_dir)

In [69]:
def match_images_labels(images, labels_dir):
    """match image and label"""
    final_pairs = []
    for img in images:
        base = os.path.splitext(os.path.basename(img))[0]
        lbl = os.path.join(labels_dir, base + ".txt")
        if os.path.exists(lbl):
            final_pairs.append((img, lbl))
        else:
            print(f"⚠️ Missing label for {img}")
    return final_pairs

### Shuffle the dataset


# Split into train and validation

In [70]:
def split_and_copy(pairs, train_ratio, output_base):
    """Split dataset into train/val and copy files"""
    shuffle(pairs)
    train_count = int(len(pairs) * train_ratio)
    train_files = pairs[:train_count]
    val_files = pairs[train_count:]

    paths = {
        "train_images": os.path.join(output_base, "train", "images"),
        "train_labels": os.path.join(output_base, "train", "labels"),
        "val_images": os.path.join(output_base, "validation", "images"),
        "val_labels": os.path.join(output_base, "validation", "labels"),
    }
    for p in paths.values():
        os.makedirs(p, exist_ok=True)

    def copy_files(file_list, img_dst, lbl_dst):
        for img, lbl in file_list:
            shutil.copy(img, os.path.join(img_dst, os.path.basename(img)))
            shutil.copy(lbl, os.path.join(lbl_dst, os.path.basename(lbl)))

    copy_files(train_files, paths["train_images"], paths["train_labels"])
    copy_files(val_files, paths["val_images"], paths["val_labels"])

    print(f"✅ Dataset ready! Total: {len(pairs)} | Train: {len(train_files)} | Val: {len(val_files)}")



# Function to copy files

In [71]:
# =======================
# Main Execution
# =======================

# Step 1: Collect files
images, labels_xml, labels_txt = gather_files(source_paths)

print(f"Found images: {len(images)}, XML labels: {len(labels_xml)}, TXT labels: {len(labels_txt)}")

# Step 2: Convert XML to TXT if needed
labels_dir = os.path.join(output_base, "all_labels")
os.makedirs(labels_dir, exist_ok=True)

if labels_xml:
    prepare_labels(labels_xml, labels_dir)

# Step 3: Copy existing TXT labels to labels_dir
for lbl in labels_txt:
    shutil.copy(lbl, os.path.join(labels_dir, os.path.basename(lbl)))

# Step 4: Match images and labels
pairs = match_images_labels(images, labels_dir)

# =======================
# Remove duplicates and ensure each image has a label
# =======================
unique_pairs = {}
for img, lbl in pairs:
    base = os.path.basename(img)
    unique_pairs[base] = (img, lbl)

pairs = list(unique_pairs.values())
print(f"After removing duplicates: {len(pairs)} valid image-label pairs")

# Step 5: Split into train/val and copy
split_and_copy(pairs, train_ratio, output_base)

Found images: 879, XML labels: 875, TXT labels: 0
⚠️ Missing label for E:\image\CarLicense\car\329.jpg
⚠️ Missing label for E:\image\CarLicense\car\330.jpg
⚠️ Missing label for E:\image\CarLicense\car\425.jpg
⚠️ Missing label for E:\image\CarLicense\car\43.jpg
After removing duplicates: 875 valid image-label pairs
✅ Dataset ready! Total: 875 | Train: 700 | Val: 175


In [72]:
import os
import yaml

# Path where yaml file will be saved
yaml_path = r'E:\image\CarLicense\licenses_cars\data.yaml'

# Dataset description for YOLO
dataset = {
    'train': r'E:/image/CarLicense/licenses_cars/train/images',
    'val': r'E:/image/CarLicense/licenses_cars/validation/images',
    'nc': 2,                # number of classes
    'names': ["car","licence"]      # class names
}

# Save yaml file
with open(yaml_path, 'w') as f:
    yaml.dump(dataset, f, sort_keys=False)

print("✅ YAML saved at:", os.path.abspath(yaml_path))


✅ YAML saved at: E:\image\CarLicense\licenses_cars\data.yaml


In [73]:
# from ultralytics import YOLO

# # Load YOLOv11 small (downloads automatically if missing)
# model = YOLO("yolov11s.pt")

# # Train on your dataset
# model.train(
#     data=r"E:\image\CarLicense\licenses_cars\data.yaml",
#     epochs=30,
#     imgsz=416,
#     batch=16,
#     device=0
# )


In [74]:
!yolo task=detect mode=train \
    model="E:/Datasets/yolo11s.pt" \
    data="E:/image/CarLicense/licenses_cars/data.yaml" \
    epochs=100 imgsz=640 batch=8 device=0 \
    verbose=False \
    save=True \
    project="E:/runs/detect" \
    name="train_yolo11s" \
    exist_ok=True \
    workers=4


New https://pypi.org/project/ultralytics/8.3.201 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.200  Python-3.13.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/image/CarLicense/licenses_cars/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=E:/Datasets/yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, n

In [75]:
# from ultralytics import YOLO

# # Load YOLOv11 small model
# model = YOLO("yolov11s.pt")

# # Train
# model.train(
#     data=r"E:\image\CarLicense\licenses_cars\data.yaml",
#     epochs=30,
#     imgsz=416,
#     batch=16,
#     device=0
# )


In [2]:
%load_ext tensorboard
%tensorboard --logdir runs

In [3]:
!kill 12936


'kill' is not recognized as an internal or external command,
operable program or batch file.


In [77]:
%reload_ext tensorboard

In [78]:
!yolo task=detect mode=predict \
    model="E:/runs/detect/train_yolo11s/weights/best.pt" \
    source="E:/image/CarLicense/test_images/" \
    imgsz=640 \
    device=0 \
    save=True \
    project="E:/runs/detect" \
    name="predict_yolo11s" \
    exist_ok=True


Ultralytics 8.3.200  Python-3.13.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs

image 1/21 E:\image\CarLicense\test_images\Cars0.png: 352x640 1 licence, 99.1ms
image 2/21 E:\image\CarLicense\test_images\Cars1.png: 416x640 1 licence, 53.5ms
image 3/21 E:\image\CarLicense\test_images\Cars2.png: 640x640 1 licence, 15.4ms
image 4/21 E:\image\CarLicense\test_images\Cars47.png: 480x640 2 licences, 49.7ms
image 5/21 E:\image\CarLicense\test_images\Cars48.png: 384x640 1 licence, 49.7ms
image 6/21 E:\image\CarLicense\test_images\Cars51.png: 448x640 1 licence, 50.3ms
image 7/21 E:\image\CarLicense\test_images\Cars52.png: 480x640 1 licence, 16.2ms
image 8/21 E:\image\CarLicense\test_images\Cars55.png: 480x640 1 licence, 14.5ms
image 9/21 E:\image\CarLicense\test_images\Cars9.png: 512x640 1 licence, 50.5ms
image 10/21 E:\image\CarLicense\test_images\Example-of-Labels-Car-make-model-year-color-